# 07 - Transfer Learning: ResNet50, EfficientNetB0, EfficientNetB3 e DenseNet121

Este notebook implementa as tarefas 15, 16, 17 e 18 da ordem recomendada.

Modelos avaliados:

1. ResNet50
2. EfficientNetB0
3. EfficientNetB3
4. DenseNet121

Fluxo de cada modelo:

1. Carregar backbone pre-treinado.
2. Substituir a cabeca final por uma saida binaria.
3. Congelar o backbone e treinar apenas a cabeca classificadora.
4. Liberar os blocos finais e fazer fine-tuning parcial.
5. Avaliar no conjunto de teste com as mesmas metricas dos demais modelos.

## Pre-requisitos

Execute antes:

1. `00_download_dataset_kaggle.ipynb`
2. `01_eda_dataset.ipynb`
3. `02_preprocessamento_splits.ipynb`
4. `03_validacao_preprocessamento_dataloaders.ipynb`
5. `04_validacao_pipeline_treino_avaliacao.ipynb`
6. `05_treinamento_cnn_propria.ipynb`
7. `06_treinamento_cnn_padrao.ipynb`

Observacao: `pretrained=True` pode baixar pesos do torchvision se eles ainda nao estiverem em cache.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import torch
from torch import nn

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.datasets import DataLoaderConfig, create_dataloaders
from src.models.transfer_learning import (
    create_transfer_model,
    get_trainable_parameter_names,
    transfer_model_summary,
    unfreeze_final_blocks,
)
from src.training.evaluate import evaluate_model
from src.training.train import TrainConfig, train_model

config.ensure_project_directories()
config.seed_everything()

print("DEVICE:", config.DEVICE)
print("SPLITS_DIR:", config.SPLITS_DIR)

## Configuracao do Experimento

Para teste rapido, reduza `MODELS_TO_RUN` e use poucas epocas. Para o experimento final, rode os quatro modelos.

In [ ]:
MODELS_TO_RUN = [
    "resnet50",
    "efficientnet_b0",
    "efficientnet_b3",
    "densenet121",
]

PRETRAINED = True
HEAD_EPOCHS = 3
FINE_TUNE_EPOCHS = 5
HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-5
DROPOUT = 0.30

loader_config = DataLoaderConfig(
    batch_size=config.BATCH_SIZE,
    num_workers=config.NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    use_weighted_sampler=False,
)

required_splits = [config.SPLITS_DIR / f"{split}.csv" for split in ["train", "val", "test"]]
missing_splits = [path for path in required_splits if not path.exists()]
if missing_splits:
    raise FileNotFoundError(
        "Splits ausentes: "
        + ", ".join(str(path) for path in missing_splits)
        + ". Execute notebooks/02_preprocessamento_splits.ipynb primeiro."
    )

loaders = create_dataloaders(dataloader_config=loader_config)
for split_name, loader in loaders.items():
    print(split_name, "imagens:", len(loader.dataset), "classes:", loader.dataset.class_counts)

## Funcao de Treino por Modelo

A funcao abaixo treina duas fases:

1. `head`: backbone congelado.
2. `finetune`: ultimos blocos liberados.

As metricas de teste sao salvas para as duas fases.

In [ ]:
def train_transfer_model(model_name: str) -> list[dict]:
    print(f"\n===== {model_name} =====")
    model = create_transfer_model(
        model_name=model_name,
        pretrained=PRETRAINED,
        freeze_backbone=True,
        dropout=DROPOUT,
    )
    print("Resumo inicial:", transfer_model_summary(model))
    print("Parametros treinaveis iniciais:", len(get_trainable_parameter_names(model)))

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=HEAD_LR,
        weight_decay=config.WEIGHT_DECAY,
    )

    head_train_config = TrainConfig(
        model_name=f"{model_name}_head",
        epochs=HEAD_EPOCHS,
        device=config.DEVICE,
        metric_to_maximize="f1",
        use_amp=torch.cuda.is_available(),
    )
    head_result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        criterion=criterion,
        optimizer=optimizer,
        train_config=head_train_config,
    )
    head_metrics = evaluate_model(
        model=model,
        dataloader=loaders["test"],
        model_name=f"{model_name}_head",
        device=config.DEVICE,
        output_dir=config.METRICS_DIR,
        checkpoint_path=head_result["best_checkpoint_path"],
    )
    head_metrics["phase"] = "head"

    unfreeze_final_blocks(model, model_name)
    print("Resumo fine-tuning:", transfer_model_summary(model))
    print("Parametros treinaveis fine-tuning:", len(get_trainable_parameter_names(model)))

    fine_tune_optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=FINE_TUNE_LR,
        weight_decay=config.WEIGHT_DECAY,
    )
    fine_tune_config = TrainConfig(
        model_name=f"{model_name}_finetune",
        epochs=FINE_TUNE_EPOCHS,
        device=config.DEVICE,
        metric_to_maximize="f1",
        use_amp=torch.cuda.is_available(),
        gradient_clip_norm=1.0,
    )
    fine_tune_result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        criterion=criterion,
        optimizer=fine_tune_optimizer,
        train_config=fine_tune_config,
    )
    fine_tune_metrics = evaluate_model(
        model=model,
        dataloader=loaders["test"],
        model_name=f"{model_name}_finetune",
        device=config.DEVICE,
        output_dir=config.METRICS_DIR,
        checkpoint_path=fine_tune_result["best_checkpoint_path"],
    )
    fine_tune_metrics["phase"] = "finetune"

    return [head_metrics, fine_tune_metrics]

## Execucao dos Modelos

Esta etapa pode demorar. No PC com RTX 5070 Ti, vale acompanhar uso de VRAM e ajustar `BATCH_SIZE` em `src/config.py` se necessario.

In [ ]:
all_metrics = []

for model_name in MODELS_TO_RUN:
    model_metrics = train_transfer_model(model_name)
    all_metrics.extend(model_metrics)

transfer_metrics_df = pd.DataFrame(all_metrics)
transfer_metrics_path = config.METRICS_DIR / "transfer_learning_metrics_summary.csv"
transfer_metrics_df.to_csv(transfer_metrics_path, index=False)
display(transfer_metrics_df)
print("Resumo parcial salvo em:", transfer_metrics_path)

## Comparacao Inicial

Esta tabela sera usada depois no notebook final de comparacao de todos os modelos.

In [ ]:
transfer_metrics_df = pd.DataFrame(all_metrics)
columns_to_show = [
    "model_name",
    "phase",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "auc_roc",
    "seconds_per_image",
    "total_parameters",
    "trainable_parameters",
    "checkpoint_size_bytes",
]
available_columns = [column for column in columns_to_show if column in transfer_metrics_df.columns]
transfer_metrics_df[available_columns].sort_values(["f1", "auc_roc"], ascending=False)

## Proxima etapa

Depois de treinar os modelos de Transfer Learning, seguir para a tarefa 19: Vision Transformer.